# Distill the kNN memory into a proxy reward model

Extract this add-on into the completed `reward_gap_followup` project. `knn_distillation` must sit beside `ppo_engine.py` and this notebook. Keep the original `inputs`, source studies, M2/M4 memories, full PPO checkpoints and original Python environment.

The teacher is the **original frozen proxy + one frozen kNN memory**. A separate proxy copy learns its corrected score through LoRA adapters and the scalar reward head. The original teacher encoder never changes. During student PPO, the reward uses only the trained student: no original-proxy forward, judge call or neighbor lookup.

The notebook runs in the **foreground**, waits until completion, and supports unchanged-configuration resume. Run only one GPU experiment at a time.

## 1. Choose the source and budget

- `SOURCE_KIND='refresh2'`: start from the completed `refresh_M2` policy, normally update 300, and use M2 as teacher memory.
- `SOURCE_KIND='refresh34'`: start from the final adaptive policy of the completed refresh3/4 study, normally update 500, and use its final memory, normally M4.

It does not choose a source silently. If selecting refresh34, finish that study first.

Default per seed: 2,400 training prompts, 400 validation prompts, 400 offline-test prompts. Each gets one new base-policy answer and one new source-policy answer: **4,800 fit examples + 800 validation examples + 800 offline examples**. These prompt splits come from the existing 3,200 refresh2 PPO training prompts and are disjoint from teacher-memory prompt groups. Offline prompts are held out from student fitting and new PPO, although the source policy has already seen them during earlier PPO.

The three PPO branches use original proxy, fixed proxy+kNN, and frozen distilled student, for 100 updates each. All restore the same per-seed source policy/value/optimizer/RNG state. New PPO uses only the distillation-training prompt split, with fresh answers. Monitoring reuses the separate refresh2 audit cohort. Final evaluation reserves 1,024 previously unused HH test groups.

Set `INCLUDE_DIRECT_JUDGE_STUDENT=True` before starting to add a fourth PPO branch whose student learns direct large-judge scores on the same fit/validation answers. It requires extra judge labels. With False, main fit/validation pseudo-labeling uses **zero new large-judge calls**; offline and policy evaluation still use the judge.

In [1]:
from pathlib import Path
import json, sys, subprocess
from IPython.display import display, FileLink, Image

PROJECT_DIR = Path.cwd()  # e.g. /workspace/reward_gap_followup
SOURCE_KIND = 'refresh2'  # change to 'refresh34' to use completed M4
FOLLOWUP_STUDY = None
REFRESH2_STUDY = None
REFRESH34_STUDY = None
SEEDS = [42]
INCLUDE_DIRECT_JUDGE_STUDENT = False
TRAIN_PROMPTS = 2400
VALIDATION_PROMPTS = 400
OFFLINE_PROMPTS = 400
FINAL_PROMPTS = 512
STUDENT_EPOCHS = 3
PPO_UPDATES = 100
ALLOW_DOWNLOADS = True
EXTRA_HF_CACHE = None
PREFLIGHT_ONLY = False

ADDON = PROJECT_DIR / 'knn_distillation'
if not (ADDON / 'run.py').is_file():
    raise FileNotFoundError('Extract the full distillation add-on into the original completed project.')
settings = json.loads((ADDON / 'settings.json').read_text())
settings.update(source_kind=SOURCE_KIND, seeds=SEEDS,
                include_direct_judge_student=INCLUDE_DIRECT_JUDGE_STUDENT,
                train_prompts=TRAIN_PROMPTS, validation_prompts=VALIDATION_PROMPTS,
                offline_prompts=OFFLINE_PROMPTS, final_prompts=FINAL_PROMPTS,
                student_epochs=STUDENT_EPOCHS, ppo_updates=PPO_UPDATES,
                allow_downloads=ALLOW_DOWNLOADS, extra_hf_cache=EXTRA_HF_CACHE)
CONFIG = ADDON / 'notebook_settings.json'
CONFIG.write_text(json.dumps(settings, indent=2) + '\n')
CMD = [sys.executable, '-u', '-m', 'knn_distillation.run', '--project', str(PROJECT_DIR.resolve()),
       '--config', str(CONFIG.resolve())]
for flag, value in [('--followup', FOLLOWUP_STUDY), ('--refresh2', REFRESH2_STUDY), ('--refresh34', REFRESH34_STUDY)]:
    if value is not None: CMD += [flag, str(Path(value).resolve())]
print(json.dumps(settings, indent=2))


{
  "source_kind": "refresh2",
  "seeds": [
    42
  ],
  "train_prompts": 2400,
  "validation_prompts": 400,
  "offline_prompts": 400,
  "final_prompts": 512,
  "ppo_updates": 100,
  "monitor_every": 50,
  "include_direct_judge_student": false,
  "student_epochs": 3,
  "student_batch_size": 4,
  "student_accumulation": 8,
  "student_lr": 2e-05,
  "student_head_lr": 2e-05,
  "student_lora_rank": 8,
  "student_lora_alpha": 16,
  "student_max_grad_norm": 1.0,
  "student_checkpoint_every": 25,
  "gradient_checkpointing": true,
  "data_seed": 2026091231,
  "student_seed": 2026091232,
  "review_pairs_per_seed": 20,
  "latency_examples": 32,
  "latency_repeats": 5,
  "allow_downloads": true,
  "extra_hf_cache": null
}


## 2. Start or resume

The RunPod preflight validates original source/input hashes, source checkpoints, encoder parity and untrained-student proxy parity. It tests a real discarded student gradient step, adapter save/load, and a discarded PPO step using only the student reward.

Stages: cache teacher labels → fit students/select epochs on validation MSE → lock selections → offline fidelity and latency → matched PPO → fresh final evaluation and blinded-review export.

The original normalization is fixed: student PPO reward is `(student_raw - original_proxy_mean) / original_proxy_std`. The student is not corrected by kNN a second time. The 256-token answer limit and whole-answer reward scoring remain fixed.

The real tiny-Qwen CPU validation includes gradient learning, bit-identical optimizer resume, save/load, all reward routes, and a full small run through reports and export. The full-size GPU experiment has not been run here. Preflight checks the actual RunPod models and checkpoints without replacing installed packages.

In [2]:
result = subprocess.run(CMD + (['--preflight'] if PREFLIGHT_ONLY else []), cwd=PROJECT_DIR)
if result.returncode:
    raise RuntimeError('Distillation experiment failed; read the error above. Original checkpoints remain available.')


GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition VRAM GiB: 95.0
All three pinned models are ready. Prompt data and adapters are bundled.


Traceback (most recent call last):
  File "/workspace/reward_gap_followup/knn_distillation/run.py", line 258, in main
    data = prepare(project, refresh2, out, c, o, assets)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/reward_gap_followup/knn_distillation/data.py", line 137, in prepare
    raise ValueError(f'Need {o["final_prompts"]} unused final HH test groups; found {len(candidates)}. No split was shrunk or reused.')
ValueError: Need 512 unused final HH test groups; found 194. No split was shrunk or reused.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/workspace/reward_gap_followup/knn_distillation/run.py", line 283, in <module>
    main()
  File "/workspace/reward_gap_followup/knn_distillation/run.py", line 258, in main
    data = prepare(project, refresh2, out, c, o, assets)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace

RuntimeError: Distillation experiment failed; read the error above. Original checkpoints remain available.

## 3. Monitor or pause from a separate terminal

```bash
python -m knn_distillation.control status --project .
python -m knn_distillation.control pause --project .
```

Wait for `stage: paused` before powering off. Rerun the start cell with unchanged settings to resume cached labels, student optimizer/cursor, PPO checkpoints and evaluation batches. Changing a scientific setting creates a separate study and may require a new final-test reservation.

The primary PPO comparison is **student minus knn at the fixed final update**. Also inspect proxy and optional judge-student results. Offline correlation alone is not the success criterion: check post-PPO student–teacher errors, judge scores, answer length/refusals and human ratings.

In [ ]:
subprocess.run([sys.executable, '-m', 'knn_distillation.control', 'status',
                '--project', str(PROJECT_DIR.resolve())], cwd=PROJECT_DIR, check=True)
latest = PROJECT_DIR / 'knn_distillation_outputs/latest.json'
if latest.exists():
    OUT = Path(json.loads(latest.read_text())['output'])
    def link(p):
        try: p = p.relative_to(PROJECT_DIR)
        except ValueError: pass
        display(FileLink(str(p)))
    for name in ['important_outcomes_knn_distillation.zip', 'distilled_reward_adapters.zip',
                 'reports/final_by_seed.csv', 'reports/paired_deltas_by_seed.csv',
                 'reports/conditional_intervals.json', 'reports/post_ppo_student_teacher_fidelity.json',
                 'reports/training_costs.csv', 'selected_students.json']:
        path = OUT / name
        if path.exists(): link(path)
    for seed in SEEDS:
        for name in [f'offline/seed_{seed}/metrics.json', f'latency/seed_{seed}/results.json']:
            path = OUT / name
            if path.exists(): link(path)
    plot = OUT / 'reports/monitoring_curves.png'
    if plot.exists(): display(Image(filename=str(plot)))


## 4. Saved students

`distilled_reward_adapters.zip` contains selected reward adapters, scalar heads, tokenizers and standalone scoring code for each seed. Use the exact original pinned proxy base checkpoint named in `reward_config.json`. A selected student's inference needs no teacher memory, judge or policy checkpoint.

`important_outcomes_knn_distillation.zip` contains reports, scores, label costs, settings, code and a blinded-review pack. Preserve the full project for exact continuation: optimizer checkpoints are excluded from compact exports. Send reviewers only the nested `final_blinded_review_BLINDED.zip`, not the full results/private key.

Latency compares identical scoring inputs and excludes diagnostic model passes from the student route. All models remain resident during the experiment's benchmark, so incremental GPU peaks are not an isolated deployment-memory measurement. Report adapter/memory bytes, training cost and repeated scoring latency before claiming an efficiency improvement.